In [1]:
import json, requests, gzip, re, os
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
print(os.getcwd())
from collections import defaultdict
from scipy.stats import binomtest
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from statsmodels.stats.multitest import multipletests

OUT_DIR = Path("../data/gdc_cache")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("data/tcga_maf")
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path("data/tcga_omics")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 8

CANCERS = [
    "TCGA-BLCA",  # Bladder Urothelial Carcinoma
    "TCGA-BRCA",  # Breast Invasive Carcinoma
    "TCGA-CESC",  # Cervical Squamous Cell Carcinoma
    "TCGA-CHOL",  # Cholangiocarcinoma
    "TCGA-COAD",  # Colon Adenocarcinoma
    "TCGA-DLBC",  # Diffuse Large B-cell Lymphoma
    "TCGA-ESCA",  # Esophageal Carcinoma
    "TCGA-GBM",   # Glioblastoma Multiforme
    "TCGA-HNSC",  # Head and Neck Squamous Cell Carcinoma
    "TCGA-KICH",  # Kidney Chromophobe
    "TCGA-KIRC",  # Kidney Renal Clear Cell Carcinoma
    "TCGA-KIRP",  # Kidney Renal Papillary Cell Carcinoma
    "TCGA-LAML",  # Acute Myeloid Leukemia
    "TCGA-LGG",   # Lower Grade Glioma
    "TCGA-LIHC",  # Liver Hepatocellular Carcinoma
    "TCGA-LUAD",  # Lung Adenocarcinoma
    "TCGA-LUSC",  # Lung Squamous Cell Carcinoma
    "TCGA-MESO",  # Mesothelioma
    "TCGA-OV",    # Ovarian Serous Cystadenocarcinoma
    "TCGA-PAAD",  # Pancreatic Adenocarcinoma
    "TCGA-PCPG",  # Pheochromocytoma and Paraganglioma
    "TCGA-PRAD",  # Prostate Adenocarcinoma
    "TCGA-READ",  # Rectum Adenocarcinoma
    "TCGA-SARC",  # Sarcoma
    "TCGA-SKCM",  # Skin Cutaneous Melanoma
    "TCGA-STAD",  # Stomach Adenocarcinoma
    "TCGA-TGCT",  # Testicular Germ Cell Tumors
    "TCGA-THCA",  # Thyroid Carcinoma
    "TCGA-THYM",  # Thymoma
    "TCGA-UCEC",  # Uterine Corpus Endometrial Carcinoma
    "TCGA-UCS",   # Uterine Carcinosarcoma
    "TCGA-UVM"    # Uveal Melanoma
]

FILE_LIMITS = {
    # "mutation": 30,
    "expression": 30,
    # "cna": 30,
    # "mirna": 15,
}


EXPR_FILE = "../data/EB++AdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.xena"
PHENO_FILE = "../data/TCGA_phenotype_denseDataOnlyDownload.tsv.gz"

COLORS = [
    '#0077B6','#0000FF','#00B4D8','#48EAC4','#F1C0E8','#B9FBC0',
    '#32CD32','#BEE1E6','#8A2BE2','#E377C2','#8EECF5','#A3C4F3',
    '#FFB347','#FFD700','#FF69B4','#CD5C5C','#7FFFD4','#FF7F50',
    '#C71585','#20B2AA','#6A5ACD','#40E0D0','#FF8C00','#DC143C',
    '#9ACD32','#1F77B4','#FF1493','#2E8B57','#D2691E','#9932CC',
    '#00CED1','#FF4500','#708090'
]


DATA_TYPES = {
    # "mutation": "Masked Somatic Mutation",
    "expression": "Gene Expression Quantification",
    # "cna": "Copy Number Segment",
    # "mirna": "miRNA Expression Quantification",
}


GDC_FILES_API = "https://api.gdc.cancer.gov/files"
GDC_DATA_API = "https://api.gdc.cancer.gov/data"


def get_latest_gencode_url():
    base = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"

    r = requests.get(base)
    r.raise_for_status()

    versions = re.findall(r"release_(\d+)", r.text)
    latest = max(map(int, versions))

    url = f"{base}/release_{latest}/gencode.v{latest}.annotation.gtf.gz"

    print(f"[INFO] Latest GENCODE: v{latest}")
    return url, f"v{latest}"

GENCODE_URL, GENCODE_VERSION = get_latest_gencode_url()
# GTF_PATH = OUT_DIR / "gencode.v38.annotation.gtf.gz"
GTF_PATH = OUT_DIR / f"gencode.{GENCODE_VERSION}.annotation.gtf.gz"

DATA_TYPES = {
    # "mutation": "Masked Somatic Mutation",
    "expression": "Gene Expression Quantification",
    # "cna": "Copy Number Segment",
    # "mirna": "miRNA Expression Quantification",
}


GDC_FILES_API = "https://api.gdc.cancer.gov/files"
GDC_DATA_API = "https://api.gdc.cancer.gov/data"


def get_latest_gencode_url():
    base = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"

    r = requests.get(base)
    r.raise_for_status()

    versions = re.findall(r"release_(\d+)", r.text)
    latest = max(map(int, versions))

    url = f"{base}/release_{latest}/gencode.v{latest}.annotation.gtf.gz"

    print(f"[INFO] Latest GENCODE: v{latest}")
    return url, f"v{latest}"

GENCODE_URL, GENCODE_VERSION = get_latest_gencode_url()
# GTF_PATH = OUT_DIR / "gencode.v38.annotation.gtf.gz"
GTF_PATH = OUT_DIR / f"gencode.{GENCODE_VERSION}.annotation.gtf.gz"

def download_gtf():
    if GTF_PATH.exists() and GTF_PATH.stat().st_size > 1e6:
        print("[cache] GTF")
        return GTF_PATH
    print("[download] GTF")
    r = requests.get(GENCODE_URL, stream=True)
    r.raise_for_status()
    with open(GTF_PATH, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    return GTF_PATH

def parse_gtf():
    path = download_gtf()

    gene_map = {}
    gene_annot = {}

    # Count total lines for tqdm
    with gzip.open(path, "rt") as f:
        total_lines = sum(1 for _ in f)

    with gzip.open(path, "rt") as f:
        for line in tqdm(f, total=total_lines, desc="Parsing GTF"):
            if line.startswith("#"):
                continue

            parts = line.split("\t")
            if parts[2] != "gene":
                continue

            chrom = parts[0]
            start = int(parts[3])
            end = int(parts[4])
            attr = parts[8]

            gid = re.search(r'gene_id "([^"]+)"', attr)
            gname = re.search(r'gene_name "([^"]+)"', attr)
            gtype = re.search(r'gene_type "([^"]+)"', attr)

            if not gid or not gname or not gtype:
                continue

            gid = gid.group(1).split(".")[0]
            gname = gname.group(1)
            gtype = gtype.group(1)

            if gtype != "protein_coding":
                continue

            gene_map[gid] = gname
            gene_annot[gname] = (chrom, start, end)

    gene_list = sorted(gene_annot.keys())

    print(f"[INFO] {len(gene_list)} genes loaded")
    return gene_list, gene_map, gene_annot



def query_files(project, data_type, size=1000):
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
            {"op": "in", "content": {"field": "data_type", "value": [data_type]}},
        ],
    }

    params = {
        "filters": json.dumps(filters),
        "format": "JSON",
        "size": str(size),
        "fields": "file_id,file_name,created_datetime",
        "sort": "created_datetime:desc"   
    }

    r = requests.get(GDC_FILES_API, params=params)
    r.raise_for_status()
    hits = r.json()["data"]["hits"]

    return [(h["file_id"], h["file_name"], h["created_datetime"]) for h in hits]

def download_one(fid, fname, subdir, min_size=1000):
    path = OUT_DIR / subdir / fname
    path.parent.mkdir(parents=True, exist_ok=True)

    # Cache check
    if path.exists():
        size = path.stat().st_size
        if size > min_size:
            print(f"[cache] {fname}")
            return path
        else:
            print(f"[redo] {fname}")
            path.unlink()

    url = f"{GDC_DATA_API}/{fid}"

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)

    return path

def parallel_download(files, subdir):
    paths = []
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futures = {ex.submit(download_one, fid, fname, subdir): fname for fid, fname in files}
        for f in as_completed(futures):
            try:
                paths.append(f.result())
            except Exception as e:
                print("[ERROR]", futures[f], e)
    return paths

GDC_API_FILES = "https://api.gdc.cancer.gov/files"
GDC_API_DATA = "https://api.gdc.cancer.gov/data"

def get_maf_file_ids(project):
    params = {
        "filters": {
            "op": "and",
            "content": [
                {"op": "in", "content": {"field": "cases.project.project_id", "value": [f"TCGA-{project}"]}},
                {"op": "in", "content": {"field": "data_type", "value": ["Masked Somatic Mutation"]}}
            ]
        },
        "fields": "file_id,file_name",
        "format": "JSON",
        "size": 200
    }

    r = requests.post(GDC_API_FILES, json=params)
    r.raise_for_status()
    hits = r.json()["data"]["hits"]
    return [(f["file_id"], f["file_name"]) for f in hits]

def download_maf_files(file_ids, project):
    project_dir = DATA_DIR / project
    project_dir.mkdir(parents=True, exist_ok=True)

    # for file_id, file_name in file_ids:
    for file_id, file_name in tqdm(file_ids, desc=f"Downloading {project}"):
        file_path = project_dir / file_name
        tmp_path = file_path.with_suffix(".tmp")

        if file_path.exists():
            print(f"[SKIP] {file_name}")
            continue

        url = f"{GDC_API_DATA}/{file_id}"
        print(f"[DOWNLOAD] {file_name}")

        try:
            r = requests.get(url, stream=True, timeout=60)

            if r.status_code != 200:
                print(f"[ERROR] Failed {file_id} ({r.status_code})")
                continue

            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            tmp_path.rename(file_path)

        except requests.exceptions.RequestException as e:
            print(f"[ERROR] {file_name}: {e}")
            if tmp_path.exists():
                tmp_path.unlink()



# def load_table(path):
#     try:
#         return pd.read_csv(path, sep="\t", comment="#", low_memory=False)
#     except Exception:
#         return pd.read_csv(path, low_memory=False)
    


GDC_API_FILES = "https://api.gdc.cancer.gov/files"
GDC_API_DATA = "https://api.gdc.cancer.gov/data"

def get_maf_file_ids(project):
    params = {
        "filters": {
            "op": "and",
            "content": [
                {"op": "in", "content": {"field": "cases.project.project_id", "value": [f"TCGA-{project}"]}},
                {"op": "in", "content": {"field": "data_type", "value": ["Masked Somatic Mutation"]}}
            ]
        },
        "fields": "file_id,file_name",
        "format": "JSON",
        "size": 200
    }

    r = requests.post(GDC_API_FILES, json=params)
    r.raise_for_status()
    hits = r.json()["data"]["hits"]
    return [(f["file_id"], f["file_name"]) for f in hits]


def download_maf_files(file_ids, project):
    project_dir = DATA_DIR / project
    project_dir.mkdir(parents=True, exist_ok=True)

    # for file_id, file_name in file_ids:
    for file_id, file_name in tqdm(file_ids, desc=f"Downloading {project}"):
        file_path = project_dir / file_name
        tmp_path = file_path.with_suffix(".tmp")

        if file_path.exists():
            print(f"[SKIP] {file_name}")
            continue

        url = f"{GDC_API_DATA}/{file_id}"
        print(f"[DOWNLOAD] {file_name}")

        try:
            r = requests.get(url, stream=True, timeout=60)

            if r.status_code != 200:
                print(f"[ERROR] Failed {file_id} ({r.status_code})")
                continue

            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            tmp_path.rename(file_path)

        except requests.exceptions.RequestException as e:
            print(f"[ERROR] {file_name}: {e}")
            if tmp_path.exists():
                tmp_path.unlink()

def compute_mutation_frequency(project):
    cancer_dir = DATA_DIR / project

    gene_sample_pairs = set()
    sample_set = set()

    for maf_file in cancer_dir.rglob("*.maf*"):
        try:
            df = pd.read_csv(
                maf_file,
                sep="\t",
                comment="#",
                usecols=["Hugo_Symbol", "Tumor_Sample_Barcode"]
            ).dropna().drop_duplicates()

            pairs = set(zip(df["Hugo_Symbol"], df["Tumor_Sample_Barcode"]))
            gene_sample_pairs.update(pairs)
            sample_set.update(df["Tumor_Sample_Barcode"])

        except:
            continue

    gene_counts = defaultdict(int)
    for gene, sample in gene_sample_pairs:
        gene_counts[gene] += 1

    n = len(sample_set)
    return pd.Series({g: c/n for g, c in gene_counts.items()})


def load_table(path):
    try:
        return pd.read_csv(path, sep="\t", comment="#", low_memory=False)
    except Exception:
        return pd.read_csv(path, low_memory=False)    
    

def minmax(s):
    s = s.fillna(0)
    return (s - s.min()) / (s.max() - s.min() + 1e-12)


def fdr_bh(pvals):
    pvals = np.asarray(pvals)
    n = len(pvals)

    order = np.argsort(pvals)
    ranked = pvals[order]

    fdr = ranked * n / (np.arange(1, n + 1))
    fdr = np.minimum.accumulate(fdr[::-1])[::-1]

    fdr_corrected = np.empty_like(fdr)
    fdr_corrected[order] = fdr

    return np.clip(fdr_corrected, 0, 1)

/Users/ericsali/Documents/2024_Winter/Project_gnn/reactome_markers/gnn_pathways/ASPIRE-GNN/process
[INFO] Latest GENCODE: v49
[INFO] Latest GENCODE: v49


In [2]:

def plot_top_genes_combined(expr_df, cancer, top_n=15):
    score = expr_df["logFC"] * -np.log10(expr_df["fdr"] + 1e-10)

    expr_df = expr_df.copy()
    expr_df["score"] = score

    top_genes = expr_df.reindex(
        expr_df["score"].abs().sort_values(ascending=False).head(top_n).index
    )

    plt.figure(figsize=(10, 8))
    plt.barh(top_genes.index, top_genes["score"])
    plt.title(f"{cancer} - Most Dysregulated Genes")
    plt.xlabel("Signed significance score")
    plt.ylabel("Gene")
    plt.gca().invert_yaxis()
    plt.show()

def plot_top_genes_logfc(expr_df, cancer, top_n=15):
    top_up = expr_df.sort_values("logFC", ascending=False).head(top_n)
    top_down = expr_df.sort_values("logFC", ascending=True).head(top_n)

    plt.figure(figsize=(10, 8))

    plt.barh(top_up.index, top_up["logFC"])
    plt.title(f"{cancer} - Top Upregulated Genes")
    plt.xlabel("logFC")
    plt.ylabel("Gene")
    plt.gca().invert_yaxis()
    plt.show()

    plt.figure(figsize=(10, 8))
    plt.barh(top_down.index, top_down["logFC"])
    plt.title(f"{cancer} - Top Downregulated Genes")
    plt.xlabel("logFC")
    plt.ylabel("Gene")
    plt.gca().invert_yaxis()
    plt.show()

def plot_volcano(expr_df, cancer):
    logFC = expr_df["logFC"]
    neglogp = -np.log10(expr_df["pval"] + 1e-10)

    plt.figure(figsize=(8, 6))
    plt.scatter(logFC, neglogp, s=10)

    plt.title(f"{cancer} Volcano Plot")
    plt.xlabel("logFC")
    plt.ylabel("-log10(p-value)")

    plt.axhline(-np.log10(0.05), linestyle="--")
    plt.axvline(1, linestyle="--")
    plt.axvline(-1, linestyle="--")

    plt.show()


def plot_heatmap_zscore(GE_all, top_n=50):
    mat = GE_all.copy()

    var = mat.var(axis=1)
    top_genes = var.sort_values(ascending=False).head(top_n).index

    mat = mat.loc[top_genes]

    # z-score per gene
    mat = (mat - mat.mean(axis=1).values[:, None]) / (mat.std(axis=1).values[:, None] + 1e-8)

    plt.figure(figsize=(12, 10))
    plt.imshow(mat.values, aspect="auto")

    plt.colorbar(label="Z-score")

    plt.xticks(range(len(mat.columns)), mat.columns, rotation=90)
    plt.yticks(range(len(mat.index)), mat.index)

    plt.title("Z-score Heatmap Across Cancers")
    plt.xlabel("Cancer")
    plt.ylabel("Gene")

    plt.tight_layout()
    plt.show()


def plot_clustered_heatmap(GE_all, top_n=50):
    mat = GE_all.copy()

    var = mat.var(axis=1)
    top_genes = var.sort_values(ascending=False).head(top_n).index

    mat = mat.loc[top_genes]

    sns.clustermap(
        mat,
        figsize=(12, 10),
        cmap="viridis",
        standard_scale=0
    )

def plot_heatmap(GE_all, top_n=50):
    mat = GE_all.copy()

    # select most variable genes (more informative)
    var = mat.var(axis=1)
    top_genes = var.sort_values(ascending=False).head(top_n).index

    mat = mat.loc[top_genes]

    plt.figure(figsize=(12, 10))
    plt.imshow(mat.values, aspect="auto")

    plt.colorbar(label="Expression score")

    plt.xticks(range(len(mat.columns)), mat.columns, rotation=90)
    plt.yticks(range(len(mat.index)), mat.index)

    plt.title("Top Variable Genes Across Cancers")
    plt.xlabel("Cancer")
    plt.ylabel("Gene")

    plt.tight_layout()
    plt.show()


In [3]:
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import os

# ================================
# FDR (Benjamini–Hochberg)



def normalize_tcga_id(s):
    if not isinstance(s, str):
        return s
    return "-".join(s.split("-")[:4])


EXPR_FILE = "../data/EB++AdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.xena"
def load_xena_expression(gene_map):
    df = pd.read_csv(EXPR_FILE, sep="\t")

    df = df.rename(columns={"sample": "gene"})
    df = df.set_index("gene")

    df.index = df.index.astype(str)

    # Detect ENSG IDs
    if df.index.str.startswith("ENSG").any():
        df.index = df.index.str.split(".").str[0]
        df.index = df.index.map(gene_map)

    df = df[~df.index.isna()]

    # collapse duplicates
    df = df.groupby(df.index).mean()

    return df

def build_sample_info_from_xena(columns):
    sample_info = {}

    for s in columns:
        parts = s.split("-")
        if len(parts) < 4:
            continue

        code = parts[3][:2]

        if code == "01":
            t = "tumor"
        elif code == "11":
            t = "normal"
        else:
            continue

        # # ✅ FIXED (closed string properly)
        # cancer = f"TCGA-{parts[1]}"

        tcga = DISEASE_TO_TCGA.get(cancer, None)

        if tcga is None:
            continue

        sample_info[s] = {
            "cancer": tcga,
            "type": t
        }

    return sample_info

def process_expression(mat, sample_info,
                       gene_list=None,
                       fdr_thresh=0.05,
                       logfc_thresh=0.5,
                       min_var=1e-5):

    print(f"[DEBUG] Input matrix: {mat.shape}")

    mat = np.log2(mat + 1)
    mat = mat.groupby(mat.index).mean()

    if gene_list is not None:
        mat = mat.reindex(gene_list)

    mat = mat.fillna(0)

    mat = mat[mat.var(axis=1) > min_var]

    print(f"[DEBUG] After variance filter: {mat.shape}")

    results = {}

    cancers = sorted(set(v["cancer"] for v in sample_info.values()))

    for cancer in cancers:

        tumor = [
            s for s, v in sample_info.items()
            if v["cancer"] == cancer and v["type"] == "tumor" and s in mat.columns
        ]

        normal = [
            s for s, v in sample_info.items()
            if v["cancer"] == cancer and v["type"] == "normal" and s in mat.columns
        ]

        print(f"[DEBUG] {cancer}: tumor={len(tumor)}, normal={len(normal)}")

        if len(tumor) < 3 or len(normal) < 3:
            print(f"[WARN] Skipping {cancer}")
            continue

        T = mat[tumor]
        N = mat[normal]

        logFC = T.mean(axis=1) - N.mean(axis=1)

        _, pvals = ttest_ind(
            T.T, N.T,
            axis=0,
            equal_var=False,
            nan_policy="omit"
        )

        pval = pd.Series(pvals, index=mat.index).fillna(1.0)
        fdr = pd.Series(fdr_bh(pval.values), index=mat.index)

        sig = ((fdr < fdr_thresh) & (np.abs(logFC) > logfc_thresh)).astype(int)

        results[cancer] = pd.DataFrame({
            "gene": mat.index,
            "logFC": logFC,
            "pval": pval,
            "fdr": fdr,
            "sig": sig
        }).sort_values("fdr")

    return results

def load_expression_cached(gene_map):
    cache_file = OUT_DIR / "xena_expression_matrix.pkl"

    if cache_file.exists():
        print("[cache] Expression matrix")
        return pd.read_pickle(cache_file)

    print("[build] Expression matrix")
    mat = load_xena_expression(gene_map)

    mat.to_pickle(cache_file)
    return mat



def process_expression_all_cached(mat, sample_info, gene_list):
    cache_file = OUTPUT_DIR / "expression_all.pkl"

    if cache_file.exists():
        print("[cache] Expression ALL")
        return pd.read_pickle(cache_file)

    print("[compute] Expression ALL")
    expr_dict = process_expression(mat, sample_info, gene_list)

    pd.to_pickle(expr_dict, cache_file)
    return expr_dict

def save_expression_per_cancer(expr_dict, cancer):

    cache_file = OUTPUT_DIR / f"{cancer}_expression_scores.csv"

    if cache_file.exists():
        print(f"[cache] Expression scores: {cancer}")
        return pd.read_csv(cache_file)

    # ✅ FIX: use full key (NO stripping)
    if cancer not in expr_dict:
        print(f"[WARN] {cancer} not found in expression results")
        return None

    df = expr_dict[cancer].copy()

    # compute score
    df["score"] = df["logFC"] * (-np.log10(df["fdr"] + 1e-10))

    df_out = df.rename(columns={
        "logFC": "mean_expr"
    })[["gene", "mean_expr", "pval", "fdr", "score"]]

    df_out.to_csv(cache_file, index=False)

    print(f"[save] {cancer}")
    return df_out


def load_xena_phenotype():
    print("[load] local phenotype")
    return pd.read_csv(PHENO_FILE, sep="\t")

DISEASE_TO_TCGA = {
    "Breast invasive carcinoma": "TCGA-BRCA",
    "Lung adenocarcinoma": "TCGA-LUAD",
    "Lung squamous cell carcinoma": "TCGA-LUSC",
    "Head and Neck squamous cell carcinoma": "TCGA-HNSC",
}
DISEASE_TO_TCGA = {
    k.lower(): v for k, v in DISEASE_TO_TCGA.items()
}

def normalize_tcga_id(s):
    return "-".join(s.split("-")[:4])


def build_sample_info_from_phenotype(columns):
    pheno = load_xena_phenotype()

    # normalize phenotype index
    pheno["sample_short"] = pheno["sample"].apply(normalize_tcga_id)
    pheno = pheno.set_index("sample_short")

    sample_info = {}

    for s in columns:
        sid = normalize_tcga_id(s)

        if sid not in pheno.index:
            continue

        row = pheno.loc[sid]

        cancer = row.get("_primary_disease", None)
        sample_type = row.get("sample_type", "")

        if pd.isna(cancer):
            continue

        if "Tumor" in sample_type:
            t = "tumor"
        elif "Normal" in sample_type:
            t = "normal"
        else:
            continue

        cancer_key = DISEASE_TO_TCGA.get(cancer.strip(), None)

        if cancer_key is None:
            continue

        sample_info[s] = {
            "cancer": cancer_key,
            "type": t
        }


    print(f"[INFO] Built sample_info: {len(sample_info)} samples")

    return sample_info
# ================================
# MAIN
# ================================
gene_list, gene_map, gene_annot = parse_gtf()

# load cached expression matrix
mat = load_expression_cached(gene_map)

sample_info = build_sample_info_from_phenotype(mat.columns)

# 🔥 ADD HERE
valid_samples = set(mat.columns) & set(sample_info.keys())
sample_info = {k: v for k, v in sample_info.items() if k in valid_samples}
mat = mat[list(valid_samples)]

# DEBUG
types = pd.Series([v["type"] for v in sample_info.values()])
cancers = pd.Series([v["cancer"] for v in sample_info.values()])

print(types.value_counts())
print(cancers.value_counts().head())

# THEN run DE
expr_dict = process_expression_all_cached(mat, sample_info, gene_list)

print(mat.columns[:10])

# # build sample info
# sample_info = build_sample_info_from_xena(mat.columns)
sample_info = build_sample_info_from_phenotype(mat.columns)

# DEBUG
types = pd.Series([v["type"] for v in sample_info.values()])
cancers = pd.Series([v["cancer"] for v in sample_info.values()])

print("Sample counts:")
print(types.value_counts())

print("\nCancer counts:")
print(cancers.value_counts().head())

# compute ALL cancers once
expr_dict = process_expression_all_cached(mat, sample_info, gene_list)

print("expr_dict keys:", list(expr_dict.keys()))

expr_saved = {}

for cancer in CANCERS:
    print(f"\n=== {cancer} ===")

    df = save_expression_per_cancer(expr_dict, cancer)

    if df is None:
        continue

    expr_saved[cancer] = df

    # plotting (reuse CNA logic)
    df_plot = df.set_index("gene")

    plot_top_genes_combined(
        df_plot.rename(columns={"score": "logFC"}),
        cancer
    )
    plot_top_genes_logfc(
        df_plot.rename(columns={"score": "logFC"}),
        cancer
    )

# ================================
# Cancer-level matrices 
# ================================
GE_pval_cancer = pd.DataFrame(index=gene_list)
GE_fdr_cancer = pd.DataFrame(index=gene_list)
GE_score_cancer = pd.DataFrame(index=gene_list)

for cancer in CANCERS:

    key = cancer  # ✅ always use full TCGA name

    if key not in expr_dict:
        print(f"[WARN] {key} missing in expr_dict")
        continue

    df = expr_dict[key].set_index("gene")
    df = df.reindex(gene_list)

    score = df["logFC"] * (-np.log10(df["fdr"] + 1e-10))

    GE_pval_cancer[cancer] = df["pval"]
    GE_fdr_cancer[cancer] = df["fdr"]
    GE_score_cancer[cancer] = score

# fill missing
GE_pval_cancer = GE_pval_cancer.fillna(1.0)
GE_fdr_cancer = GE_fdr_cancer.fillna(1.0)
GE_score_cancer = GE_score_cancer.fillna(0)

# DEBUG
print("Columns:", GE_pval_cancer.columns)

# save
GE_pval_cancer.to_csv(OUTPUT_DIR / "expression_pvalues_per_cancer.csv")
GE_fdr_cancer.to_csv(OUTPUT_DIR / "expression_fdr_per_cancer.csv")
GE_score_cancer.to_csv(OUTPUT_DIR / "expression_scores_per_cancer.csv")





for cancer, df in expr_dict.items():
    print(f"\nPlotting {cancer}")

    df = df.set_index("gene")

    plot_top_genes_combined(df, cancer=cancer)
    plot_top_genes_logfc(df, cancer=cancer)
    plot_volcano(df, cancer)
    plot_heatmap_zscore(df, top_n=50)
    # plot_clustered_heatmap(df, top_n=50)

# ================================
# Save per-cancer expression (CNA-style)
# ================================
EXPR_OUT_DIR = OUTPUT_DIR / "expression_per_cancer"
EXPR_OUT_DIR.mkdir(parents=True, exist_ok=True)


GE_all = pd.DataFrame(index=gene_list)
GE_pval_all = pd.DataFrame(index=gene_list)
GE_fdr_all = pd.DataFrame(index=gene_list)

for cancer in CANCERS:

    # ✅ use full key
    if cancer not in expr_dict:
        print(f"[WARN] {cancer} missing in expr_dict")
        continue

    df = expr_dict[cancer].set_index("gene")
    df = df.reindex(gene_list)

    score = df["logFC"] * (-np.log10(df["fdr"] + 1e-8))

    GE_all[cancer] = score
    GE_pval_all[cancer] = df["pval"]
    GE_fdr_all[cancer] = df["fdr"]

# ================================
# Filtering
# ================================
sig_mask = (GE_fdr_all < 0.05)
support_mask = (GE_all.abs() > 0)

mask = sig_mask.any(axis=1) | support_mask.any(axis=1)

print(f"[INFO] Significant genes: {sig_mask.any(axis=1).sum()}")
print(f"[INFO] Supported genes: {support_mask.any(axis=1).sum()}")

GE_all = GE_all.loc[mask]
GE_pval_all = GE_pval_all.loc[mask]
GE_fdr_all = GE_fdr_all.loc[mask]

print(f"[FINAL] Kept {mask.sum()} genes")

# ================================
# Save
# ================================
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

GE_all.to_csv(OUTPUT_DIR / "expression_features.csv")
GE_pval_all.to_csv(OUTPUT_DIR / "expression_pvalues.csv")
GE_fdr_all.to_csv(OUTPUT_DIR / "expression_fdr.csv")

print("✅ Saved outputs")

[cache] GTF


Parsing GTF:   0%|          | 0/7750159 [00:00<?, ?it/s]

[INFO] 20070 genes loaded
[build] Expression matrix


FileNotFoundError: [Errno 2] No such file or directory: '../data/EB++AdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.xena'

In [ ]:
!pip install statsmodels

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ------------- -------------------------- 3.1/9.6 MB 18.5 MB/s eta 0:00:01
   --------------------------- ------------ 6.6/9.6 MB 18.3 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 18.1 MB/s eta 0:00:00


In [ ]:
# %%
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# ================================
# PATHS
# ================================
SAVE_DIR = "data/tcga_expression"
os.makedirs(SAVE_DIR, exist_ok=True)

EXPR_FILE = "../data/EB++AdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.xena"
PHENO_FILE = "../data/TCGA_phenotype_denseDataOnlyDownload.tsv.gz"

# ================================
# DISEASE MAP
# ================================
disease_to_tcga = {
    "adrenocortical carcinoma": "ACC",
    "bladder urothelial carcinoma": "BLCA",
    "breast invasive carcinoma": "BRCA",
    "cervical squamous cell carcinoma and endocervical adenocarcinoma": "CESC",
    "cholangiocarcinoma": "CHOL",
    "colon adenocarcinoma": "COAD",
    "rectum adenocarcinoma": "READ",
    "diffuse large b-cell lymphoma": "DLBC",
    "esophageal carcinoma": "ESCA",
    "glioblastoma multiforme": "GBM",
    "head and neck squamous cell carcinoma": "HNSC",
    "kidney chromophobe": "KICH",
    "kidney renal clear cell carcinoma": "KIRC",
    "kidney renal papillary cell carcinoma": "KIRP",
    "acute myeloid leukemia": "LAML",
    "brain lower grade glioma": "LGG",
    "liver hepatocellular carcinoma": "LIHC",
    "lung adenocarcinoma": "LUAD",
    "lung squamous cell carcinoma": "LUSC",
    "mesothelioma": "MESO",
    "ovarian serous cystadenocarcinoma": "OV",
    "pancreatic adenocarcinoma": "PAAD",
    "pheochromocytoma and paraganglioma": "PCPG",
    "prostate adenocarcinoma": "PRAD",
    "sarcoma": "SARC",
    "skin cutaneous melanoma": "SKCM",
    "stomach adenocarcinoma": "STAD",
    "testicular germ cell tumors": "TGCT",
    "thyroid carcinoma": "THCA",
    "thymoma": "THYM",
    "uterine corpus endometrial carcinoma": "UCEC",
    "uterine carcinosarcoma": "UCS",
    "uveal melanoma": "UVM"
}

DISEASE_TO_TCGA = {
    "Breast invasive carcinoma": "TCGA-BRCA",
    "Lung adenocarcinoma": "TCGA-LUAD",
    "Lung squamous cell carcinoma": "TCGA-LUSC",
    "Head and Neck squamous cell carcinoma": "TCGA-HNSC",
}

# ================================
# CLEAN GENE NAMES
# ================================
def clean_gene_column(df):
    df["gene"] = df["gene"].astype(str).str.split(".").str[0]
    df = df[~df["gene"].str.match(r"^\d+$")]
    df = df[df["gene"].notna()]
    df = df[df["gene"] != ""]
    df["gene"] = df["gene"].str.upper()
    return df

# ================================
# LOAD DATA
# ================================
print("[LOAD] phenotype")
pheno = pd.read_csv(PHENO_FILE, sep="\t", compression="gzip")
pheno.columns = pheno.columns.str.strip()

pheno["Sample"] = pheno["sample"]
pheno["_primary_disease"] = pheno["_primary_disease"].str.strip().str.lower()

# tumor vs normal
pheno["is_tumor"] = pheno["sample_type"].str.contains("tumor", case=False, na=False)

# map cancer
pheno["Cancer"] = pheno["_primary_disease"].map(disease_to_tcga)
pheno = pheno.dropna(subset=["Cancer"])

print("[LOAD] expression")
expr = pd.read_csv(EXPR_FILE, sep="\t")
expr.rename(columns={expr.columns[0]: "gene"}, inplace=True)
expr = clean_gene_column(expr)

# ================================
# ALIGN SAMPLES
# ================================
common_samples = list(set(expr.columns) & set(pheno["Sample"]))

expr = expr[["gene"] + common_samples]
pheno = pheno[pheno["Sample"].isin(common_samples)]

print("Matched samples:", len(common_samples))

# ================================
# BUILD SAMPLE MATRIX
# ================================
X_sample = expr.set_index("gene").T  # (samples × genes)

# normalize TCGA IDs
def short_barcode(s):
    return s[:12]

pheno["short"] = pheno["Sample"].apply(short_barcode)
X_sample["short"] = X_sample.index.map(short_barcode)

# merge phenotype
X_sample = X_sample.merge(
    pheno[["short", "Cancer", "is_tumor"]],
    on="short",
    how="inner"
)

print("Aligned samples:", X_sample.shape)

# ================================
# DIFFERENTIAL EXPRESSION
# ================================
def compute_expression_DE(X_sample, genes):

    results = {}

    cancers = sorted(X_sample["Cancer"].unique())

    for cancer in cancers:
        df = X_sample[X_sample["Cancer"] == cancer]

        tumor = df[df["is_tumor"] == True]
        normal = df[df["is_tumor"] == False]

        print(f"[DEBUG] {cancer}: tumor={len(tumor)}, normal={len(normal)}")

        if len(tumor) < 5 or len(normal) < 5:
            print(f"[WARN] Skipping {cancer}")
            continue

        T = tumor.drop(columns=["short", "Cancer", "is_tumor"])
        N = normal.drop(columns=["short", "Cancer", "is_tumor"])

        # log transform
        T = np.log2(T + 1)
        N = np.log2(N + 1)

        # logFC
        logFC = T.mean(axis=0) - N.mean(axis=0)

        # t-test
        _, pvals = ttest_ind(
            T.values,
            N.values,
            axis=0,
            equal_var=False,
            nan_policy="omit"
        )

        pval = pd.Series(pvals, index=T.columns).fillna(1.0)

        # FDR
        _, fdr_vals, _, _ = multipletests(pval.values, method="fdr_bh")
        fdr = pd.Series(fdr_vals, index=T.columns)

        # score
        score = logFC * (-np.log10(fdr + 1e-10))

        df_res = pd.DataFrame({
            "gene": T.columns,
            "logFC": logFC,
            "pval": pval,
            "fdr": fdr,
            "score": score
        }).sort_values("fdr")

        results[cancer] = df_res

    return results

genes = expr["gene"].values
expr_dict = compute_expression_DE(X_sample, genes)

print("Cancers computed:", list(expr_dict.keys()))

# ================================
# SAVE PER-CANCER
# ================================
for cancer, df in expr_dict.items():
    out_file = os.path.join(SAVE_DIR, f"{cancer}_expression_scores.csv")
    df.to_csv(out_file, index=False)
    print("Saved:", out_file)

# ================================
# BUILD MATRICES (GNN READY)
# ================================
GE_score = pd.DataFrame(index=genes)
GE_pval = pd.DataFrame(index=genes)
GE_fdr = pd.DataFrame(index=genes)

for cancer, df in expr_dict.items():

    # 🔥 FIX: remove duplicates
    df = df.groupby("gene", as_index=False).mean()

    df = df.set_index("gene")
    df = df.reindex(genes)

    GE_score[cancer] = df["score"]
    GE_pval[cancer] = df["pval"]
    GE_fdr[cancer] = df["fdr"]

# fill missing
GE_score = GE_score.fillna(0)
GE_pval = GE_pval.fillna(1.0)
GE_fdr = GE_fdr.fillna(1.0)

# save
GE_score.to_csv(os.path.join(SAVE_DIR, "expression_scores_per_cancer.csv"))
GE_pval.to_csv(os.path.join(SAVE_DIR, "expression_pvalues_per_cancer.csv"))
GE_fdr.to_csv(os.path.join(SAVE_DIR, "expression_fdr_per_cancer.csv"))

print("✅ Saved all outputs")

[LOAD] phenotype
[LOAD] expression
Matched samples: 8266
Aligned samples: (9458, 20505)
[DEBUG] BLCA: tumor=427, normal=38
[DEBUG] BRCA: tumor=1218, normal=248
[DEBUG] CHOL: tumor=45, normal=18
[DEBUG] COAD: tumor=496, normal=83


c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log2
  result = func(self.values, **kwargs)
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log2
  result = func(self.values, **kwargs)
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_stats_py.py:1079: RuntimeWarning: invalid value encountered in subtract
  a_zero_mean = a - mean
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_stats_py.py:6527: RuntimeWarning: invalid value encountered in scalar subtract
  d = mean1 - mean2
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_stats_py.py:7025: RuntimeWarning: invalid value encountered in scalar subtract
  estimate = m1-m2


[DEBUG] DLBC: tumor=48, normal=0
[WARN] Skipping DLBC
[DEBUG] ESCA: tumor=196, normal=24


c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log2
  result = func(self.values, **kwargs)
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log2
  result = func(self.values, **kwargs)
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_stats_py.py:1079: RuntimeWarning: invalid value encountered in subtract
  a_zero_mean = a - mean
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_stats_py.py:6527: RuntimeWarning: invalid value encountered in scalar subtract
  d = mean1 - mean2
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_stats_py.py:7025: RuntimeWarning: invalid value encountered in scalar subtract
  estimate = m1-m2


[DEBUG] GBM: tumor=179, normal=5
[DEBUG] KICH: tumor=91, normal=50
[DEBUG] LAML: tumor=0, normal=173
[WARN] Skipping LAML
[DEBUG] LGG: tumor=558, normal=0
[WARN] Skipping LGG
[DEBUG] LIHC: tumor=427, normal=100
[DEBUG] LUAD: tumor=579, normal=117
[DEBUG] LUSC: tumor=553, normal=102
[DEBUG] MESO: tumor=87, normal=0
[WARN] Skipping MESO
[DEBUG] OV: tumor=314, normal=0
[WARN] Skipping OV
[DEBUG] PAAD: tumor=183, normal=10
[DEBUG] PRAD: tumor=550, normal=106


...

[DEBUG] READ: tumor=172, normal=19


c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log2
  result = func(self.values, **kwargs)
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log2
  result = func(self.values, **kwargs)
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_stats_py.py:1079: RuntimeWarning: invalid value encountered in subtract
  a_zero_mean = a - mean
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_stats_py.py:6527: RuntimeWarning: invalid value encountered in scalar subtract
  d = mean1 - mean2
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_stats_py.py:7025: RuntimeWarning: invalid value encountered in scalar subtract
  estimate = m1-m2


[DEBUG] SARC: tumor=271, normal=6
[DEBUG] SKCM: tumor=105, normal=377
[DEBUG] STAD: tumor=447, normal=67


c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log2
  result = func(self.values, **kwargs)
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log2
  result = func(self.values, **kwargs)
C:\Users\erics\AppData\Local\Temp\ipykernel_20140\699752561.py:160: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  _, pvals = ttest_ind(
c:\Users\erics\miniconda3\envs\reactome_gnn\Lib\site-packages\scipy\stats\_axis_nan_policy.py:621: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return result_to_tuple(hypotest_fun_out(*samp

[DEBUG] THCA: tumor=572, normal=134
[DEBUG] THYM: tumor=122, normal=4
[WARN] Skipping THYM
[DEBUG] UCS: tumor=57, normal=0
[WARN] Skipping UCS
[DEBUG] UVM: tumor=80, normal=0
[WARN] Skipping UVM
Cancers computed: ['BLCA', 'BRCA', 'CHOL', 'COAD', 'ESCA', 'GBM', 'KICH', 'LIHC', 'LUAD', 'LUSC', 'PAAD', 'PRAD', 'READ', 'SARC', 'SKCM', 'STAD', 'THCA']
Saved: data/tcga\BLCA_expression_scores.csv
Saved: data/tcga\BRCA_expression_scores.csv
Saved: data/tcga\CHOL_expression_scores.csv
Saved: data/tcga\COAD_expression_scores.csv
Saved: data/tcga\ESCA_expression_scores.csv
Saved: data/tcga\GBM_expression_scores.csv
Saved: data/tcga\KICH_expression_scores.csv
Saved: data/tcga\LIHC_expression_scores.csv
Saved: data/tcga\LUAD_expression_scores.csv
Saved: data/tcga\LUSC_expression_scores.csv
Saved: data/tcga\PAAD_expression_scores.csv
Saved: data/tcga\PRAD_expression_scores.csv
Saved: data/tcga\READ_expression_scores.csv
Saved: data/tcga\SARC_expression_scores.csv
Saved: data/tcga\SKCM_expression_sco